# Geneformer and scGPT for Single-Cell Modeling

**Tier 5 — Modern AI for Science | Module 07 · Notebook 1**

*Prerequisites: Single-cell preprocessing basics, transformer fundamentals*

---

**By the end of this notebook you will be able to:**
1. Explain rank-based tokenization for single-cell profiles
2. Build toy cell embeddings from expression matrices
3. Perform simple cell-type annotation from embedding space
4. Prototype perturbation-response prediction workflow

---

[← Module 06](../06_Protein_Language_Models/) | [Module Overview](README.md)

---

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(17)

## 1. Toy expression matrix

We create a synthetic matrix (cells × genes) with three cell types and marker genes.

In [ ]:
genes = [f'G{i:02d}' for i in range(30)]
n_cells = 240

types = np.random.choice(['Tcell', 'Bcell', 'Myeloid'], size=n_cells, p=[0.35, 0.30, 0.35])
X = np.random.gamma(shape=1.5, scale=1.0, size=(n_cells, len(genes)))

markers = {
    'Tcell': [1, 2, 3],
    'Bcell': [8, 9, 10],
    'Myeloid': [15, 16, 17],
}

for i, t in enumerate(types):
    X[i, markers[t]] += np.random.uniform(2.5, 4.0)

expr = pd.DataFrame(X, columns=genes)
expr.head(3)

## 2. Rank-token style embedding

Geneformer-style intuition: represent each cell by ranked genes rather than raw count vectors.

In [ ]:
def topk_rank_tokens(row: np.ndarray, k: int = 8):
    idx = np.argsort(row)[::-1][:k]
    return tuple(idx.tolist())

tokens = [topk_rank_tokens(expr.iloc[i].values, k=8) for i in range(n_cells)]

def toy_cell_embedding(token_tuple, n_genes=30):
    v = np.zeros(n_genes, dtype=float)
    for rank, g in enumerate(token_tuple):
        v[g] += 1.0 / (rank + 1)
    return v / (np.linalg.norm(v) + 1e-9)

E = np.vstack([toy_cell_embedding(t, n_genes=len(genes)) for t in tokens])
print('Embedding matrix:', E.shape)

## 3. Cell-type annotation via nearest centroid

In [ ]:
y = np.array(types)
idx = np.arange(n_cells)
np.random.shuffle(idx)
train = idx[:180]
test = idx[180:]

centroids = {c: E[train][y[train] == c].mean(axis=0) for c in np.unique(y)}

def predict_one(vec):
    d = {c: np.linalg.norm(vec - centroids[c]) for c in centroids}
    return min(d, key=d.get)

pred = np.array([predict_one(E[i]) for i in test])
acc = float((pred == y[test]).mean())
print('Annotation accuracy:', round(acc, 3))

## 4. Perturbation prediction prototype

scGPT-like use case: estimate post-perturbation expression from baseline profile and perturbation metadata.

In [ ]:
def perturb_predict(cell_vec: np.ndarray, pert: str) -> np.ndarray:
    out = cell_vec.copy()
    if pert == 'KO_G02':
        out[2] *= 0.2
    elif pert == 'KO_G09':
        out[9] *= 0.2
    elif pert == 'CYTOKINE_X':
        out[[1, 3, 15]] *= 1.3
    return out

cell0 = expr.iloc[0].values
for p in ['KO_G02', 'KO_G09', 'CYTOKINE_X']:
    pred_expr = perturb_predict(cell0, p)
    print(p, 'delta_mean=', round(float((pred_expr - cell0).mean()), 4))

## 5. Scaling to atlas data

For real projects, use CellxGene Census and model APIs for million-cell scale workflows; keep local prototypes for logic validation and QC.

## Summary

- Rank-based representations are effective for cell-state encoding.
- Embedding-space centroid methods give strong annotation baselines.
- Perturbation prediction adds causal/extrapolative capability beyond annotation.